![image.png](attachment:44d2f751-c0ff-48be-8e86-77b4f69628a6.png)

#### Passo 1: Instalar bibliotecas

In [1]:
#!pip install requests
#!pip install beautifulsoup4
#!pip install sqlite3
#!pip install pandas

ERROR: Could not find a version that satisfies the requirement sqlite3 (from versions: none)
ERROR: No matching distribution found for sqlite3


#### Passo 2: Importar bibliotecas

In [2]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import pandas as pd

#### Passo 3: Fromatar link de busca

Neste exemplo utilizaremos um site público de testes de scraping

http://books.toscrape.com (feito exatamente para treinar scraping).

In [3]:
# URL base
url = "https://books.toscrape.com/catalogue/category/books_1/page-1.html"

#### Passo 4: Determinar variáveis que para armazenamento
Neste exemplo, iremos utilizar:
- Título
- Preços
- Estoque

In [4]:
titulos, precos, estoque = [], [], []

#### Passo 5: Obter informações do link

In [5]:
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

#### Passo 5.1: Obter informações dos produtos

In [6]:
# Cada produto está dentro de article.product_pod
livros = soup.find_all("article", class_="product_pod")

In [7]:
livros[0].h3.a['title']

'A Light in the Attic'

In [8]:
# Preço
livros[0].find("p", class_="price_color").text.replace("Â£", "")

'51.77'

In [9]:
# Estoque
livros[0].find("p", class_="instock availability").text.strip()
# A função strip()em Python serve para remover espaços em branco ou caracteres específicos do início e/ou fim de uma string

'In stock'

#### Passo 6: Coletar dados em escala
Neste exemplo, vamos coletar dados das 5 primeiras páginas

In [10]:
for page in range(1, 6):
    pagina_url = f"https://books.toscrape.com/catalogue/category/books_1/page-{page}.html"
    response = requests.get(pagina_url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Cada produto está dentro de article.product_pod
    livros = soup.find_all("article", class_="product_pod")

    for livro in livros:
        titulo = livro.h3.a["title"]
        preco = livro.find("p", class_="price_color").text.strip()
        preco = preco.replace("Â£", "").replace("£", "")
        situacao_estoque = livro.find("p", class_="instock availability").text.strip()

        titulos.append(titulo)
        precos.append(float(preco))
        estoque.append(situacao_estoque)

print(f"Livros coletados: {len(titulos)}")


Livros coletados: 100


#### Passo 7: Criar DataFrame

In [11]:
df = pd.DataFrame({"titulo": titulos, "preco": precos, "estoque": estoque})
df.head()

,titulo,preco,estoque
0,A Light in the Attic,51.77,In stock
1,Tipping the Velvet,53.74,In stock
2,Soumission,50.10,In stock
3,Sharp Objects,47.82,In stock
4,Sapiens: A Brief History of Humankind,54.23,In stock


#### Passo 8: Criar Banco de Dados

In [12]:
# Conectar ao banco SQLite (o arquivo será criado se não existir)
conexao = sqlite3.connect("livraria.db")
cursor = conexao.cursor()


#### Passo 9: DDL – Data Definition Language
DDL serve para **definir a estrutura do banco de dados**: criar, alterar e excluir tabelas.

- `CREATE` → cria tabelas e estruturas  
- `ALTER` → modifica a estrutura de tabelas  
- `DROP` → apaga tabelas ou bancos de dados  


In [13]:
# Criar tabela para armazenar os livros coletados
cursor.execute("""
CREATE TABLE IF NOT EXISTS livros (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    titulo TEXT NOT NULL,
    preco REAL NOT NULL,
    estoque TEXT NOT NULL
)
""")
conexao.commit()


##### 9.1: Listar todas tabelas do banco
Neste exemplo iremos utilizaremos a sintaxe `SELECT name FROM sqlite_master;`

- `sqlite_master` é uma tabela especial **interna** do SQLite que guarda **metadados** sobre o banco de dados: tabelas, índices, views e triggers
- `SELECT name` seleciona apenas o nome (`name`) dos objetos armazenados no `sqlite_master`

In [14]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
cursor.fetchall()


[('livros',), ('sqlite_sequence',)]

Como pegar os resultados usando o método `fetch` (buscar)
- `fetchone()` Retorna apenas uma linha do resultado (a próxima disponível). Se não houver mais nada, retorna `None`
- `fetchmany(n)` Retorna as próximas n linhas do resultado.
- `fetchall()` Retorna todas as linhas restantes do resultado em uma lista de tuplas.

##### 9.2: Verificar o esquema da tabela (estrutura das colunas)

In [15]:
cursor.execute("PRAGMA table_info(livros);")
cursor.fetchall()


[(0, 'id', 'INTEGER', 0, None, 1),
 (1, 'titulo', 'TEXT', 1, None, 0),
 (2, 'preco', 'REAL', 1, None, 0),
 (3, 'estoque', 'TEXT', 1, None, 0)]

##### 9.3: Alterar tabela
Nesta etapa utilizaremos o comando `ALTER TABLE` informar a alteração da tabela e o comando `ADD COLUMN` para especificar a alteração

In [16]:
cursor.execute("ALTER TABLE livros ADD COLUMN categoria TEXT;")
conexao.commit()


##### 9.4: Verifica o equema da tabela após incluir coluna coluna

In [17]:
cursor.execute("PRAGMA table_info(livros);")
cursor.fetchall()


[(0, 'id', 'INTEGER', 0, None, 1),
 (1, 'titulo', 'TEXT', 1, None, 0),
 (2, 'preco', 'REAL', 1, None, 0),
 (3, 'estoque', 'TEXT', 1, None, 0),
 (4, 'categoria', 'TEXT', 0, None, 0)]

##### 9.5: Remover uma coluna
Nesta etapa utilizaremos o comando `ALTER TABLE` informar a alteração da tabela e o comando `DROP COLUMN` para especificar a alteração

In [18]:
# SQLite permite DROP COLUMN em versões recentes (SQLite 3.35.0+)
cursor.execute("ALTER TABLE livros DROP COLUMN categoria;")
conexao.commit()


##### 9.6: Verifica o equema da tabela após incluir coluna coluna

In [19]:
cursor.execute("PRAGMA table_info(livros);")
cursor.fetchall()


[(0, 'id', 'INTEGER', 0, None, 1),
 (1, 'titulo', 'TEXT', 1, None, 0),
 (2, 'preco', 'REAL', 1, None, 0),
 (3, 'estoque', 'TEXT', 1, None, 0)]

##### 9.7: Excluir tabela
Primeiro vamos criar uma tabela adicional

In [20]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS tabela_teste (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    descricao TEXT
)
""")
conexao.commit()


##### Listar todas tabelas do banco

In [21]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
cursor.fetchall()


[('livros',), ('sqlite_sequence',), ('tabela_teste',)]

##### Remover tabela do banco

In [22]:
cursor.execute("DROP TABLE IF EXISTS tabela_teste;")
conexao.commit()


##### Listar todas tabelas do banco

In [23]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
cursor.fetchall()


[('livros',), ('sqlite_sequence',)]

#### Passo 10: DML – Data Manipulation Language

DML serve para **manipular os dados** dentro das tabelas:

- `INSERT` → inserir registros  
- `UPDATE` → atualizar registros  
- `DELETE` → excluir registros



##### 10.1 `INSERT` (inserir dados)

In [24]:
# Inserir os livros coletados em lote, evitando duplicações pelo título
cursor.executemany(
    "INSERT INTO livros (titulo, preco, estoque) VALUES (?, ?, ?)",
    list(zip(titulos, precos, estoque))
)
conexao.commit()


##### Verifica quantos registros foram inseridos

In [25]:
cursor.execute("SELECT COUNT(*) FROM livros;")
cursor.fetchone()


(100,)

##### Verifica uma amostra dos registros inseridos

In [26]:
cursor.execute("SELECT id, titulo, preco, estoque FROM livros LIMIT 10;")
cursor.fetchall()


[(1, 'A Light in the Attic', 51.77, 'In stock'),
 (2, 'Tipping the Velvet', 53.74, 'In stock'),
 (3, 'Soumission', 50.1, 'In stock'),
 (4, 'Sharp Objects', 47.82, 'In stock'),
 (5, 'Sapiens: A Brief History of Humankind', 54.23, 'In stock'),
 (6, 'The Requiem Red', 22.65, 'In stock'),
 (7, 'The Dirty Little Secrets of Getting Your Dream Job', 33.34, 'In stock'),
 (8,
  'The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull',
  17.93,
  'In stock'),
 (9,
  'The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics',
  22.6,
  'In stock'),
 (10, 'The Black Maria', 52.15, 'In stock')]

##### 10.2: `UPDATE` (atualizar dados)

In [27]:
# Exemplo: aplicar um desconto de 10% aos livros com preço acima de 50
cursor.execute("UPDATE livros SET preco = preco * 0.90 WHERE preco > 50;")
conexao.commit()


##### Verifica uma amostra dos registros inseridos

In [28]:
cursor.execute("SELECT titulo, preco FROM livros WHERE preco > 50 LIMIT 10;")
cursor.fetchall()


[('Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991',
  51.525),
 ('Slow States of Collapse: Poems', 51.579),
 ('The Secret of Dreadwillow Carse', 50.517),
 ('The Pioneer Woman Cooks: Dinnertime: Comfort Classics, Freezer Food, 16-Minute Meals, and Other Delicious Ways to Solve Supper!',
  50.769),
 ('The Past Never Ends', 50.85),
 ('The Electric Pencil: Drawings from Inside State Hospital No. 3', 50.454),
 ('The Death of Humanity: and the Case for Life', 52.299),
 ('Masks and Shadows', 50.76)]

##### 10.3: Inserir um novo registro no banco

In [29]:
cursor.execute(
    "INSERT INTO livros (titulo, preco, estoque) VALUES (?, ?, ?)",
    ("Livro de exemplo", 25.90, "In stock")
)
conexao.commit()


##### Verifica quantos registros existem no banco

In [30]:
cursor.execute("SELECT COUNT(*) FROM livros;")
cursor.fetchone()


(101,)

##### Verifica uma amostra dos últimos registros inseridos

In [31]:
cursor.execute("SELECT id, titulo, preco, estoque FROM livros ORDER BY id DESC LIMIT 5;")
cursor.fetchall()


[(101, 'Livro de exemplo', 25.9, 'In stock'),
 (100, 'In the Country We Love: My Family Divided', 22.0, 'In stock'),
 (99, 'Join', 35.67, 'In stock'),
 (98,
  'Judo: Seven Steps to Black Belt (an Introductory Guide for Beginners)',
  48.51,
  'In stock'),
 (97,
  'Layered: Baking, Building, and Styling Spectacular Cakes',
  40.11,
  'In stock')]

##### 10.4: `DELETE` (exclui registros)

In [32]:
# Excluir somente o registro de exemplo inserido acima
cursor.execute("DELETE FROM livros WHERE titulo = ?;", ("Livro de exemplo",))
conexao.commit()


##### Verifica uma amostra dos últimos registros inseridos

In [33]:
cursor.execute("SELECT id, titulo, preco, estoque FROM livros ORDER BY id DESC LIMIT 5;")
cursor.fetchall()


[(100, 'In the Country We Love: My Family Divided', 22.0, 'In stock'),
 (99, 'Join', 35.67, 'In stock'),
 (98,
  'Judo: Seven Steps to Black Belt (an Introductory Guide for Beginners)',
  48.51,
  'In stock'),
 (97,
  'Layered: Baking, Building, and Styling Spectacular Cakes',
  40.11,
  'In stock'),
 (96,
  'Lumberjanes Vol. 3: A Terrible Plan (Lumberjanes #9-12)',
  19.92,
  'In stock')]

#### Passo 11: DQL serve para **consultar dados** no banco:

- `SELECT`  
- `WHERE`  
- `ORDER BY`  
- `GROUP BY`  
- Funções de agregação (`COUNT`, `AVG`, `MAX`, `MIN`, `SUM`)

##### Consulta simples com `fetchone`

In [34]:
cursor.execute("SELECT titulo, preco FROM livros LIMIT 1;")
cursor.fetchone()


('A Light in the Attic', 46.593)

##### Consulta simples com `fetchmany()`

In [35]:
cursor.execute("SELECT titulo, preco FROM livros LIMIT 5;")
cursor.fetchmany(3)


[('A Light in the Attic', 46.593),
 ('Tipping the Velvet', 48.366),
 ('Soumission', 45.09)]

##### Consulta simples com `fetchall()`

In [36]:
cursor.execute("SELECT titulo, preco FROM livros LIMIT 5;")
cursor.fetchall()


[('A Light in the Attic', 46.593),
 ('Tipping the Velvet', 48.366),
 ('Soumission', 45.09),
 ('Sharp Objects', 47.82),
 ('Sapiens: A Brief History of Humankind', 48.806999999999995)]

#### `WHERE`
##### Consulta com filtro: Livros com preço abaixo de 20

In [37]:
cursor.execute("SELECT titulo, preco FROM livros WHERE preco < 20 ORDER BY preco;")
cursor.fetchall()


[('Patience', 10.16),
 ('In Her Wake', 12.84),
 ('Princess Between Worlds (Wide-Awake Princess #5)', 13.34),
 ('Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Princess Jellyfish 2-in-1 Omnibus #1)',
  13.61),
 ('Starving Hearts (Triangular Trade Trilogy, #1)', 13.99),
 ('Mama Tried: Traditional Italian Cooking for the Screwed, Crude, Vegan, and Tattooed',
  14.02),
 ('On a Midnight Clear', 14.07),
 ('Untitled Collection: Sabbath Poems 2014', 14.27),
 ('Obsidian (Lux #1)', 14.86),
 ('Outcast, Vol. 1: A Darkness Surrounds Him (Outcast #1)', 15.44),
 ("Sophie's World", 15.94),
 ('Tsubasa: WoRLD CHRoNiCLE 2 (Tsubasa WoRLD CHRoNiCLE #2)', 16.28),
 ('The Life-Changing Magic of Tidying Up: The Japanese Art of Decluttering and Organizing',
  16.77),
 ('Thirst', 17.27),
 ('Set Me Free', 17.46),
 ('The Four Agreements: A Practical Guide to Personal Freedom', 17.66),
 ('The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull',
  17.93),
 ('Maude (1883-1993):She Grew U

#### `ORDER BY`
##### Consulta com ordenação: Top 5 livros mais caros

In [38]:
cursor.execute("SELECT titulo, preco FROM livros ORDER BY preco DESC LIMIT 5;")
cursor.fetchall()


[('The Death of Humanity: and the Case for Life', 52.299),
 ('Slow States of Collapse: Poems', 51.579),
 ('Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991',
  51.525),
 ('The Past Never Ends', 50.85),
 ('The Pioneer Woman Cooks: Dinnertime: Comfort Classics, Freezer Food, 16-Minute Meals, and Other Delicious Ways to Solve Supper!',
  50.769)]

#### `GROUP BY`
##### Consulta com agregação: Quantidade de livros por situação de estoque

In [39]:
cursor.execute("""
SELECT estoque, COUNT(*) AS quantidade
FROM livros
GROUP BY estoque
ORDER BY quantidade DESC;
""")
cursor.fetchall()


[('In stock', 100)]

#### `DISTINCT`
##### Consulta com filtro: Contagem de dados distintos

In [40]:
cursor.execute("SELECT COUNT(DISTINCT estoque) FROM livros;")
cursor.fetchone()


(1,)

#### `HAVING COUNT(*)`
##### Consulta com filtro: Contagem de dados repetidos

In [41]:
cursor.execute("""
SELECT titulo, COUNT(*) AS quantidade
FROM livros
GROUP BY titulo
HAVING COUNT(*) > 1;
""")
cursor.fetchall()


[]

#### `AVG`
##### Consulta com agregação: Valor médio dos livros

In [42]:
cursor.execute("SELECT AVG(preco) FROM livros;")
cursor.fetchone()


(33.32121,)

##### Consulta com agregação: Valor médio dos livros  (com arredondamento)

In [43]:
cursor.execute("SELECT ROUND(AVG(preco), 2) FROM livros;")
cursor.fetchone()


(33.32,)

#### `MAX`
##### Consulta com agregação: Maior valor entre os livros `MAX`

In [44]:
cursor.execute("SELECT MAX(preco) FROM livros;")
cursor.fetchone()


(52.299,)

#### `MIN`
##### Consulta com agregação: Menor valor entre os livros

In [45]:
cursor.execute("SELECT MIN(preco) FROM livros;")
cursor.fetchone()


(10.16,)

#### `SUM`
##### Consulta com agregação: Soma do valor dos livros

In [46]:
cursor.execute("SELECT SUM(preco) FROM livros;")
cursor.fetchone()


(3332.121,)

##### Consulta com agregação: Soma do valor dos livros em estoque

In [47]:
cursor.execute("""
SELECT SUM(preco)
FROM livros
WHERE LOWER(estoque) LIKE '%in stock%';
""")
cursor.fetchone()


(3332.121,)

##### Consulta com agregação: Soma do valor dos livros sem estoque

In [48]:
cursor.execute("""
SELECT SUM(preco)
FROM livros
WHERE LOWER(estoque) LIKE '%out of stock%';
""")
cursor.fetchone()


(None,)

##### Consulta de multiplos valores: Valor Máximo, Medio e Mínimo

In [49]:
cursor.execute("SELECT MAX(preco), ROUND(AVG(preco), 2), MIN(preco) FROM livros;")
cursor.fetchone()


(52.299, 33.32, 10.16)

##### Consulta com Ranking por Preço

In [50]:
cursor.execute("""
SELECT titulo, preco,
       RANK() OVER (ORDER BY preco DESC) AS ranking
FROM livros
ORDER BY preco DESC
LIMIT 10;
""")
cursor.fetchall()


[('The Death of Humanity: and the Case for Life', 52.299, 1),
 ('Slow States of Collapse: Poems', 51.579, 2),
 ('Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991',
  51.525,
  3),
 ('The Past Never Ends', 50.85, 4),
 ('The Pioneer Woman Cooks: Dinnertime: Comfort Classics, Freezer Food, 16-Minute Meals, and Other Delicious Ways to Solve Supper!',
  50.769,
  5),
 ('Masks and Shadows', 50.76, 6),
 ('The Secret of Dreadwillow Carse', 50.517, 7),
 ('The Electric Pencil: Drawings from Inside State Hospital No. 3', 50.454, 8),
 ('Birdsong: A Story in Pictures', 49.176, 9),
 ('The Bulletproof Diet: Lose up to a Pound a Day, Reclaim Energy and Focus, Upgrade Your Life',
  49.05,
  10)]

#### 12: JOINS – Relacionando tabelas

JOINS permitem **combinar dados de várias tabelas**.  
Vamos criar outras tabelas para exemplificar.

##### 12.1 Criar tabela de vendas

In [51]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS vendas (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    livro_id INTEGER NOT NULL,
    quantidade INTEGER NOT NULL,
    valor_total REAL NOT NULL,
    FOREIGN KEY (livro_id) REFERENCES livros(id)
)
""")
conexao.commit()


##### Verifica o esquema da tabela (estrutura das colunas)

In [52]:
cursor.execute("PRAGMA table_info(vendas);")
cursor.fetchall()


[(0, 'id', 'INTEGER', 0, None, 1),
 (1, 'livro_id', 'INTEGER', 1, None, 0),
 (2, 'quantidade', 'INTEGER', 1, None, 0),
 (3, 'valor_total', 'REAL', 1, None, 0)]

##### 12.2: Inserir dados de exemplo

In [53]:
# Inserir algumas vendas usando IDs de livros existentes
cursor.execute("SELECT id, preco FROM livros ORDER BY id LIMIT 3;")
livros_exemplo = cursor.fetchall()

vendas_exemplo = [
    (livro_id, 2, round(preco * 2, 2))
    for livro_id, preco in livros_exemplo
]
cursor.executemany(
    "INSERT INTO vendas (livro_id, quantidade, valor_total) VALUES (?, ?, ?)",
    vendas_exemplo
)
conexao.commit()


##### Verifica dados inseridos

In [54]:
cursor.execute("SELECT * FROM vendas;")
cursor.fetchall()


[(1, 1, 2, 93.19), (2, 2, 2, 96.73), (3, 3, 2, 90.18)]

##### 12.3: Exemplos de JOINs com a tabela existente

In [55]:
# INNER JOIN: somente livros que possuem venda
cursor.execute("""
SELECT l.titulo, v.quantidade, v.valor_total
FROM livros AS l
INNER JOIN vendas AS v ON l.id = v.livro_id;
""")
cursor.fetchall()


[('A Light in the Attic', 2, 93.19),
 ('Tipping the Velvet', 2, 96.73),
 ('Soumission', 2, 90.18)]

In [56]:
# LEFT JOIN: todos os livros, inclusive os que não possuem venda
cursor.execute("""
SELECT l.titulo, v.quantidade, v.valor_total
FROM livros AS l
LEFT JOIN vendas AS v ON l.id = v.livro_id
ORDER BY l.titulo;
""")
cursor.fetchall()


[('#HigherSelfie: Wake Up Your Life. Free Your Soul. Find Your Tribe.',
  None,
  None),
 ('A Light in the Attic', 2, 93.19),
 ('Aladdin and His Wonderful Lamp', None, None),
 ("America's Cradle of Quarterbacks: Western Pennsylvania's Football Factory from Johnny Unitas to Joe Montana",
  None,
  None),
 ('Behind Closed Doors', None, None),
 ('Birdsong: A Story in Pictures', None, None),
 ('Black Dust', None, None),
 ('Chase Me (Paris Nights #2)', None, None),
 ('Foolproof Preserving: A Guide to Small Batch Jams, Jellies, Pickles, Condiments, and More: A Foolproof Guide to Making Small Batch Jams, Jellies, Pickles, Condiments, and More',
  None,
  None),
 ('How Music Works', None, None),
 ('In Her Wake', None, None),
 ('In a Dark, Dark Wood', None, None),
 ('In the Country We Love: My Family Divided', None, None),
 ("It's Only the Himalayas", None, None),
 ('Join', None, None),
 ('Judo: Seven Steps to Black Belt (an Introductory Guide for Beginners)',
  None,
  None),
 ('Layered: Bakin

In [57]:
# JOIN com agregação: total vendido por livro
cursor.execute("""
SELECT l.titulo,
       COALESCE(SUM(v.quantidade), 0) AS unidades_vendidas,
       COALESCE(ROUND(SUM(v.valor_total), 2), 0) AS faturamento
FROM livros AS l
LEFT JOIN vendas AS v ON l.id = v.livro_id
GROUP BY l.id, l.titulo
ORDER BY faturamento DESC;
""")
cursor.fetchall()


[('Tipping the Velvet', 2, 96.73),
 ('A Light in the Attic', 2, 93.19),
 ('Soumission', 2, 90.18),
 ('Sharp Objects', 0, 0),
 ('Sapiens: A Brief History of Humankind', 0, 0),
 ('The Requiem Red', 0, 0),
 ('The Dirty Little Secrets of Getting Your Dream Job', 0, 0),
 ('The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull',
  0,
  0),
 ('The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics',
  0,
  0),
 ('The Black Maria', 0, 0),
 ('Starving Hearts (Triangular Trade Trilogy, #1)', 0, 0),
 ("Shakespeare's Sonnets", 0, 0),
 ('Set Me Free', 0, 0),
 ("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 0, 0),
 ('Rip it Up and Start Again', 0, 0),
 ('Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991',
  0,
  0),
 ('Olio', 0, 0),
 ('Mesaerion: The Best Science Fiction Stories 1800-1849', 0, 0),
 ('Libertarianism for Beginners', 0, 0),
 ("It's Only the Himalayas", 0, 0),
 ('In Her

In [58]:
# Livros cujo total de vendas foi maior que zero
cursor.execute("""
SELECT l.titulo, SUM(v.quantidade) AS unidades_vendidas
FROM livros AS l
JOIN vendas AS v ON l.id = v.livro_id
GROUP BY l.id, l.titulo
HAVING SUM(v.quantidade) > 0;
""")
cursor.fetchall()


[('A Light in the Attic', 2), ('Tipping the Velvet', 2), ('Soumission', 2)]

In [59]:
# Consulta de vendas com preço unitário e total
cursor.execute("""
SELECT l.titulo, l.preco AS preco_unitario,
       v.quantidade, v.valor_total
FROM vendas AS v
JOIN livros AS l ON l.id = v.livro_id
ORDER BY v.id;
""")
cursor.fetchall()


[('A Light in the Attic', 46.593, 2, 93.19),
 ('Tipping the Velvet', 48.366, 2, 96.73),
 ('Soumission', 45.09, 2, 90.18)]

#### 13: Transações – BEGIN, COMMIT e ROLLBACK

Uma transação é um conjunto de operações que devem ser executadas juntas.  

- `BEGIN` → inicia a transação  
- `COMMIT` → confirma as alterações  
- `ROLLBACK` → desfaz as alterações (caso de erro)  

In [ ]:
# Exemplo de transação: inserir e depois desfazer uma venda de teste
try:
    cursor.execute("BEGIN")
    cursor.execute("""
        INSERT INTO vendas (livro_id, quantidade, valor_total)
        VALUES (?, ?, ?)
    """, (livros_exemplo[0][0], 1, livros_exemplo[0][1]))
    # Desfaz a operação de exemplo para demonstrar ROLLBACK
    conexao.rollback()
except sqlite3.Error as erro:
    conexao.rollback()
    print(f"Erro na transação: {erro}")


#### Passo 13: Encerra conexão com banco de dados

In [61]:
cursor.close()
conexao.close()
